In [1]:
from utils import *

print("=== [1] Constructing Random Code Instance ===")
q = 2             # Base field characteristic (F_2)
m = 4             # Field extension degree (F_{2^4})
n = 7             # Total code length 
k = 3             # Dimension of the code
r = 2             # Target rank weight of the error vector

n_minus_k = n - k 
Fq = GF(q)
Fqm = GF(q**m, name='z')
V_Fq = Fqm.vector_space()

print(f"Parameters: q={q}, m={m}, n={n}, k={k}, r={r}\n")

# Generate a random Parity-Check matrix H with full row rank
while True:
    H = matrix(Fqm, n_minus_k, n, [Fqm.random_element() for _ in range(n_minus_k * n)])
    if H.rank() == n_minus_k:
        break

# Generate a secret error vector with exact rank r
field_basis = [Fqm.gen()**i for i in range(m)]
support_basis = random.sample(field_basis, r) # Sample r linearly independent elements

e_list = []
for _ in range(n):
    coeffs = [Fq.random_element() for _ in range(r)]
    entry = sum(int(coeffs[i]) * support_basis[i] for i in range(r))
    e_list.append(entry)
e_secret = vector(Fqm, e_list)

# 1. Construct the explicit m x n coordinate Matrix E over the base field Fq
# Each column of E corresponds to the vector representation of e_secret[j]
E_matrix = matrix(Fq, [V_Fq(el) for el in e_secret]).transpose()
assert E_matrix.rank() == r, "Error vector rank mismatch."

print(f"Generated Secret Vector e:\n{e_secret}")
print(f"Underlying Support Matrix E (dimensions {m}x{n}):\n{E_matrix}\n")

# Compute Syndrome
s = H * e_secret

# =====================================================================
# 2. CHOOSE RANDOM COLUMNS FROM THE MATRIX E TO FORM ERASURES
# =====================================================================
num_erasures = 2
J = sorted(random.sample(range(n), num_erasures))

a_J = []
for j in J:
    # Extract the j-th column from the base field matrix E
    column_vector_Fq = E_matrix.column(j)
    
    # Reconstruct the extension field element from its Fq column coordinates
    field_element_Fqm = sum(column_vector_Fq[i] * field_basis[i] for i in range(m))
    a_J.append(Fqm(field_element_Fqm))

print("=== [2] Extracting Correct Column Erasure Hints from Matrix E ===")
print(f"Selected Column Indices (J): {J}")
for idx, j in enumerate(J):
    print(f"  -> Column {j} from matrix E: {list(E_matrix.column(j))} maps to field element: {a_J[idx]}")
print(f"\nFinal a_J array passed to decoder: {a_J}\n")

# =====================================================================
# 3. RUN DECODER AND VERIFY
# =====================================================================
print("=== [3] Running Full-Entry AP Decoder Module ===")
candidate_list = full_entry_ap_decoder_module(H, s, J, a_J, q, m, n, k)
print(f"Number of candidates returned: {len(candidate_list)}")

# Final Verification
success = False
for idx, candidate in enumerate(candidate_list):
    syndrome_check = (H * candidate == s)
    cand_matrix = matrix(Fq, [V_Fq(el) for el in candidate]).transpose()
    rank_check = (cand_matrix.rank() == r)
    
    if syndrome_check and rank_check:
        print(f"[+] Candidate #{idx+1} passes validation filters.")
        if candidate == e_secret:
            print("[++++] SUCCESS: Reconstructed vector matches the secret error exactly!")
            success = True
            break

if not success:
    print("[-] FAILURE: Reconstructed target vector could not be recovered.")

=== [1] Constructing Random Code Instance ===
Parameters: q=2, m=4, n=7, k=3, r=2

Generated Secret Vector e:
(0, z, z^2, z, z^2 + z, z^2 + z, 0)
Underlying Support Matrix E (dimensions 4x7):
[0 0 0 0 0 0 0]
[0 1 0 1 1 1 0]
[0 0 1 0 1 1 0]
[0 0 0 0 0 0 0]

=== [2] Extracting Correct Column Erasure Hints from Matrix E ===
Selected Column Indices (J): [4, 5]
  -> Column 4 from matrix E: [0, 1, 1, 0] maps to field element: z^2 + z
  -> Column 5 from matrix E: [0, 1, 1, 0] maps to field element: z^2 + z

Final a_J array passed to decoder: [z^2 + z, z^2 + z]

=== [3] Running Full-Entry AP Decoder Module ===


/usr/lib/python3/dist-packages/sage/categories/fields.py:533: DeprecationWarning: The default value for map will be changing to True.  To keep the current behavior, explicitly pass map=False.
See https://trac.sagemath.org/28481 for details.
  return self.free_module(*args, **kwds)


Number of candidates returned: 9
[+] Candidate #1 passes validation filters.
[++++] SUCCESS: Reconstructed vector matches the secret error exactly!


In [3]:
# =====================================================================
# JUPYTER NOTEBOOK TEST PIPELINE FOR GRS SCALAR ERASURES DECODER
# =====================================================================

# Import the decoder directly from your utils.py file
from utils import *
import random

# 1. Configure standard instance variables
q = 2  # Base field size
m = 4  # Extension degree (GF(2^4))
n = 7  # Code length
k = 3  # Code dimension
d = 3  # Candidate super-support space dimension

Fq = GF(q)
Fqm.<z> = GF(q^m)
V_space = Fqm.vector_space(map=False)

print("====================================================")
print("     STEP 1: GENERATING SIMULATION INSTANCE         ")
print("====================================================")

# 2. Build a random instance parity-check matrix H
H = matrix(Fqm, n - k, n, lambda i, j: Fqm.random_element())

# 3. Create a clean mock support to generate verifiable target data
support_basis = []
while len(support_basis) < d:
    elem = Fqm.random_element()
    if elem != 0 and V_space(elem) not in VectorSpace(Fq, m).subspace([V_space(b) for b in support_basis]):
        support_basis.append(elem)

# 4. Generate a compatible reference error vector from our support
mock_error_coords = []
for j in range(n):
    coord_element = sum(Fq(random.randint(0, q-1)) * support_basis[lambda_] for lambda_ in range(d))
    mock_error_coords.append(coord_element)
e_mock = vector(Fqm, mock_error_coords)

# Calculate corresponding syndrome vector s
s = H * e_mock

# Derive the true rank weight of our generated mock error vector
true_rank_matrix = matrix(Fq, [V_space(el) for el in e_mock]).transpose()
r = true_rank_matrix.rank()

print(f"Parity-Check Matrix H ({H.nrows()}x{H.ncols()}):\n{H}\n")
print(f"Syndrome Vector s    : {s}")
print(f"Secret Target e_mock : {e_mock}")
print(f"Target Rank Weight r : {r}\n")

print("====================================================")
print("     STEP 2: PREPARING SIDE-CHANNEL HINTS           ")
print("====================================================")

# 5. Formulate Side-Channel Scalar Erasure Leakage Matrix (T and a_T)
T = [(0, 1), (3, 2)]  # Coordinate positions leaked: (column_index, bit_position)
a_T = {}
for (j, ell) in T:
    true_bit_val = V_space(e_mock[j])[ell]
    a_T[(j, ell)] = true_bit_val

for (j, ell) in T:
    print(f"  Side-Information Hint -> Column {j}, Bit Position {ell} = {a_T[(j, ell)]}")
print()

print("====================================================")
print("     STEP 3: RUNNING DECODER PIPELINE              ")
print("====================================================")

# 6. Execute your grs_scalar_erasures function from utils
E_F, equations, beta_matrix, R_poly = grs_scalar_erasures(
    q, m, n, d, support_basis, Fqm, H, s, T, a_T, r, B_rec=128
)

print("====================================================")
print("     STEP 4: RUNNING VERIFICATION HOOK              ")
print("====================================================")

# 7. Use the verification function defined above to isolate valid answers
valid_solutions = verify_decoder_candidates(E_F, H, s, r, Fqm)

# Final success verification
if e_mock in valid_solutions:
    print("[STATUS: PASSED] The secret target error vector was successfully recovered and verified!")
else:
    print("[STATUS: FAILED] The secret target vector was missed by the decoder or filtering process.")

     STEP 1: GENERATING SIMULATION INSTANCE         
Parity-Check Matrix H (4x7):
[z^3 + z^2 + z + 1 z^3 + z^2 + z + 1 z^3 + z^2 + z + 1           z^3 + z           z^2 + z       z^2 + z + 1                 1]
[z^3 + z^2 + z + 1 z^3 + z^2 + z + 1           z^3 + z             z + 1     z^3 + z^2 + 1     z^3 + z^2 + 1 z^3 + z^2 + z + 1]
[                z             z + 1           z^3 + 1               z^3               z^2           z^3 + 1               z^3]
[                1       z^3 + z + 1       z^3 + z + 1                 z z^3 + z^2 + z + 1               z^3 z^3 + z^2 + z + 1]

Syndrome Vector s    : (z^2 + z, z^3 + z^2, 0, z^3 + z + 1)
Secret Target e_mock : (z^3, z^3 + z^2 + z, z^2 + z, z^3 + z, z^2, z^2, z^3 + z)
Target Rank Weight r : 3

     STEP 2: PREPARING SIDE-CHANNEL HINTS           
  Side-Information Hint -> Column 0, Bit Position 1 = 0
  Side-Information Hint -> Column 3, Bit Position 2 = 0

     STEP 3: RUNNING DECODER PIPELINE              
[+] Step 8: System h